In [1]:
# =================
# 설치 + 캐시 자동화
# =================
import subprocess

# 설치
subprocess.run(['pip', 'install', 'transformers==5.11.0', 'peft==0.14.0', 
                'trl==0.12.0', 'datasets', 'accelerate', 'bitsandbytes',
                '--cache-dir', '/workspace/pip_cache', '-q'])

# torchao 제거
subprocess.run(['pip', 'uninstall', 'torchao', '-y'])

import os
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"

print("설치 완료")

s->datasets) (1.16.0)



In [6]:
# ======================
# 데이터 로드 + 토크나이저
# ======================

import json, torch, glob
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

BASE_MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"
DATA_PATH = "/workspace/workit_sft_merged_v2.json"
OUTPUT_DIR = "/workspace/workit_model_output_v2"

with open(DATA_PATH, encoding="utf-8") as f:
    raw = json.load(f)
print(f"데이터 로드: {len(raw)}개")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

def to_text(item):
    return {"text": tokenizer.apply_chat_template(
        item["messages"], tokenize=False, add_generation_prompt=False,
    )}

dataset = Dataset.from_list([to_text(d) for d in raw])
print(f"데이터셋: {len(dataset)}개")

패턴 못 찾음: /workspace/hf_cache/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py
패턴 못 찾음: /workspace/hf_cache/hub/models--LGAI-EXAONE--EXAONE-3.5-2.4B-Instruct/snapshots/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py


In [2]:
# =========================
# 모델 로드 + 캐시 자동 수정
# =========================
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# 모델 로드 후 즉시 캐시 파일 수정
OLD = """        causal_mask = create_causal_mask(
            config=self.config,
            input_embeds=inputs_embeds,
            attention_mask=attention_mask,
            cache_position=cache_position,
            past_key_values=past_key_values,
            position_ids=position_ids,
        )"""
NEW = """        causal_mask = create_causal_mask(
            config=self.config,
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
        )"""

paths = glob.glob("/workspace/hf_cache/**/modeling_exaone.py", recursive=True)
paths += glob.glob("/root/.cache/huggingface/**/modeling_exaone.py", recursive=True)
for path in paths:
    with open(path, "r") as f:
        content = f.read()
    if OLD in content:
        # pyc 삭제
        import subprocess
        subprocess.run(['find', '/workspace/hf_cache', '-name', '*.pyc', '-delete'])
        subprocess.run(['find', '/root/.cache/huggingface', '-name', '*.pyc', '-delete'])
        content = content.replace(OLD, NEW)
        with open(path, "w") as f:
            f.write(content)
        print(f"수정 완료: {path}")

print("모델 로드 완료")

데이터 로드: 782개


In [3]:
# ===========
# LoRA + 학습
# ===========

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=1024,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"학습 완료 → {OUTPUT_DIR}")

데이터셋: 782개

샘플:
[|system|]당신은 공공 SW 계약서의 위험 조항을 탐지하는 전문가입니다. 주어진 계약 조항과 참고 기준을 바탕으로 위험 여부를 판단하고 근거를 제시하십시오.[|endofturn|]
[|user|]다음 계약 조항의 위험 여부를 판단하세요.

[계약조항]
을이 계약기간 내에 용역을 완성하지 못한 경우 지체일수마다 계약금액의 0.1%를 지체상금으로 납부하여야 한다. 단, 지체상금의 총액은 계약금액의 30%를 초과할 수 없다.

[참고기준]
지방자치단체 용역계약 일반조건 제7절 제1항 가(행안부 예규 제324호): 계약서에 정한 지연배상금율에 계약금액을 곱하여 산출한 금액을 납부하여야 한다.
지방계약법 시행령 제90조③: 지연배상금 총액은 계약금액의 100분의 30을 초과할 수 없다.
[|assistant|


In [4]:
# ====
# 압축
# ====
import shutil
shutil.make_archive("/workspace/workit_model_output_v2", "zip", OUTPUT_DIR)
print("압축 완료: /workspace/workit_model_output_v2.zip")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /workspace/hf_cache/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /workspace/hf_cache/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

모델 로드 완료


In [8]:
import json
from collections import Counter

with open("/workspace/workit_sft_merged_v1.json", encoding="utf-8") as f:
    data = json.load(f)

cnt = Counter()
for d in data:
    verdict = d["messages"][2]["content"].split("\n")[0].replace("판정:", "").strip()
    cnt[verdict] += 1

total = len(data)
for k, v in cnt.most_common():
    print(f"{k}: {v}개 ({v/total*100:.1f}%)")

충족: 150개 (28.4%)
미흡: 150개 (28.4%)
위반: 70개 (13.3%)
정상: 65개 (12.3%)
누락: 61개 (11.6%)
불가: 32개 (6.1%)
